In [ ]:
import pandas as pd
import os
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

pd.set_option('display.notebook_repr_html', True)

def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, custom_means=None):
    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
    unique_vals = sorted(data[x].unique())
    palette = colors[:len(unique_vals)]
    
    sns.boxplot(data=data, x=x, y=y, ax=ax, palette=palette,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    # Get the actual order of categories as they appear in the plot
    plot_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    
    if custom_means is not None:
        # Use custom means if provided, mapping them to the correct plot positions
        for i, plot_label in enumerate(plot_labels):
            # Convert plot label back to original category name
            reverse_mapping = {
                'Anthropic': 'anthropic',
                'OpenAI': 'openai', 
                'Google': 'google',
                'DeepSeek': 'deepseek',
                'Common': 'common',
                'Complex': 'complex',
                'Script': 'script',
                'Transcript': 'transcript', 
                'Tanenbaum': 'tanenbaum',
                'Script (Manipulated)': 'script_manipulated',
                'No Source': 'no_source',
                'Exp 1a': '1a',
                'Exp 1b': '1b'
            }
            original_category = reverse_mapping.get(plot_label, plot_label.lower())
            
            if original_category in custom_means.index:
                mean_val = custom_means[original_category]
                ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                          edgecolor='darkred', linewidth=1)
    else:
        # Use original behavior if no custom means provided
        for i, val in enumerate(unique_vals):
            mean_val = data[data[x] == val][y].mean()
            ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                      edgecolor='darkred', linewidth=1)
    
    label_mapping = {
        'anthropic': 'Anthropic',
        'openai': 'OpenAI', 
        'google': 'Google',
        'deepseek': 'DeepSeek',
        'common': 'Common',
        'complex': 'Complex',
        'script': 'Script',
        'transcript': 'Transcript', 
        'tanenbaum': 'Tanenbaum',
        'script_manipulated': 'Script (Manipulated)',
        'no_source': 'No Source',
        '1a': 'Exp 1a',
        '1b': 'Exp 1b'
    }
    
    current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    new_labels = [label_mapping.get(label, label) for label in current_labels]
    ax.set_xticklabels(new_labels, rotation=0)
    
    if 'cosine_similarity' in y.lower():
        ylabel = 'Cosine Similarity'
        if '(-1 to 1)' not in title:
            title = title.replace('Cosine Similarity', 'Cosine Similarity (-1 to 1)')
    elif 'adherence_score' in y.lower():
        ylabel = 'Adherence Score'
        if '(0 to 1)' not in title:
            title = title.replace('Adherence Score', 'Adherence Score (0 to 1)')
    
    if x.lower() == 'llm':
        xlabel = 'LLM'
    elif x.lower() == 'prompt_type':
        xlabel = 'Prompt Type'
    elif x.lower() == 'input_source':
        xlabel = 'Input Source'
    elif x.lower() == 'layer':
        xlabel = 'Layer'
    elif x.lower() == 'experiment':
        xlabel = 'Experiment'
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)

BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

exp1a_path = os.path.join(
    BASE_PROJECT_PATH, "20_experiments/60_analyses/csv_files/quantitative/exp1a.csv"
)
exp1b_path = os.path.join(
    BASE_PROJECT_PATH, "20_experiments/60_analyses/csv_files/quantitative/exp1b.csv"
)

exp1a_no_source_path = os.path.join(
    BASE_PROJECT_PATH, "20_experiments/60_analyses/csv_files/quantitative/exp1a_no_source.csv"
)

output_dir_exp1a = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/tables/exp1a"
)
output_dir_exp1b = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/tables/exp1b"
)
output_plot_dir_exp1a = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/plots/exp1a"
)
output_plot_dir_exp1b = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/plots/exp1b"
)

tables = {}
plots = {}

for path in [output_dir_exp1a, output_dir_exp1b, output_plot_dir_exp1a, output_plot_dir_exp1b]:
    os.makedirs(path, exist_ok=True)

if not os.path.exists(exp1a_path) or not os.path.exists(exp1b_path):
    print("Error: Quantitative data files not found.")
    print(f"Looked for {exp1a_path} and {exp1b_path}")
else:
    exp1a_df = pd.read_csv(exp1a_path)
    exp1b_df = pd.read_csv(exp1b_path)
    
    # Load exp1a_no_source if available
    exp1a_no_source_df = None
    if os.path.exists(exp1a_no_source_path):
        exp1a_no_source_df = pd.read_csv(exp1a_no_source_path)
        exp1a_no_source_df["adherence_score"] = exp1a_no_source_df[
            ["adherence_score_openai", "adherence_score_anthropic"]
        ].mean(axis=1)
        exp1a_no_source_df = exp1a_no_source_df.drop(
            columns=["adherence_score_openai", "adherence_score_anthropic"]
        )
        print("Data loaded and preprocessed for exp1a, exp1b, and exp1a_no_source.")
    else:
        print("Warning: exp1a_no_source data not found. Some plots may be skipped.")
        print("Data loaded and preprocessed for exp1a and exp1b only.")

    exp1a_df["adherence_score"] = exp1a_df[
        ["adherence_score_openai", "adherence_score_anthropic"]
    ].mean(axis=1)
    exp1b_df["adherence_score"] = exp1b_df[
        ["adherence_score_openai", "adherence_score_anthropic"]
    ].mean(axis=1)

    exp1a_df = exp1a_df.drop(
        columns=["adherence_score_openai", "adherence_score_anthropic"]
    )
    exp1b_df = exp1b_df.drop(
        columns=["adherence_score_openai", "adherence_score_anthropic"]
    )

In [ ]:
# Add exp1a_no_source path
exp1a_no_source_path = os.path.join(
    BASE_PROJECT_PATH, "20_experiments/60_analyses/csv_files/quantitative/exp1a_no_source.csv"
)

# Add output directories for no_source
output_dir_exp1a_no_source = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/tables/exp1a_no_source"
)
output_plot_dir_exp1a_no_source = os.path.join(
    BASE_PROJECT_PATH, "40_evaluation/exp1/quantitative/plots/exp1a_no_source"
)

for path in [output_dir_exp1a_no_source, output_plot_dir_exp1a_no_source]:
    os.makedirs(path, exist_ok=True)

## Experiment 1a

### 1.1: Metrics by LLM

In [ ]:
llm_metrics_1a = exp1a_df.groupby("llm")[["cosine_similarity", "adherence_score"]].agg(
    ["mean", "std"]
)
tables["exp1a_metrics_by_llm"] = llm_metrics_1a
display(llm_metrics_1a)

### 1.2: Metrics by Input Source

In [ ]:
input_source_metrics_1a = exp1a_df.groupby("input_source")[
    ["cosine_similarity", "adherence_score"]
].agg(["mean", "std"])
tables["exp1a_metrics_by_input_source"] = input_source_metrics_1a
display(input_source_metrics_1a)

### 1.3: Metrics by Prompt Type

In [ ]:
prompt_type_metrics_1a = exp1a_df.groupby("prompt_type")[
    ["cosine_similarity", "adherence_score"]
].agg(["mean", "std"])
tables["exp1a_metrics_by_prompt_type"] = prompt_type_metrics_1a
display(prompt_type_metrics_1a)

### 1.4: Metrics by Layer

In [ ]:
layer_metrics_1a = exp1a_df.groupby("layer")[["cosine_similarity", "adherence_score"]].agg(
    ["mean", "std"]
)
tables["exp1a_metrics_by_layer"] = layer_metrics_1a
display(layer_metrics_1a)

In [ ]:
import os
import glob
import re
import pandas as pd

def extract_info_from_path(file_path):
    parts = file_path.split(os.sep)
    
    input_source = None
    layer = None
    llm = None
    prompt_type = None
    
    for i, part in enumerate(parts):
        if part in ['common_prompt', 'complex_prompt']:
            prompt_type = 'common' if part == 'common_prompt' else 'complex'
            if i + 1 < len(parts):
                llm = parts[i + 1]
            if i + 2 < len(parts):
                input_source = parts[i + 2]
            if i + 3 < len(parts):
                filename = parts[i + 3]
                if 'layer' in filename:
                    layer_match = re.search(r'layer(\d+)', filename)
                    if layer_match:
                        layer = int(layer_match.group(1))
            break
    
    return input_source, layer, llm, prompt_type

def find_free_text_questions_with_scores():
    free_text_data = []
    
    base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/10_exp1")
    all_files = glob.glob(os.path.join(base_path, "**/*.txt"), recursive=True)
    
    for file_path in all_files:
        if "_question.txt" in file_path:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                if "Antwort:" in content and "Antwortmöglichkeiten:" not in content:
                    input_source, layer, llm_file, prompt_type = extract_info_from_path(file_path)
                    
                    llm_mapping = {
                        'anthropic': 'anthropic',
                        'openai': 'openai', 
                        'google': 'google',
                        'deepseek': 'deepseek'
                    }
                    
                    llm = llm_mapping.get(llm_file, llm_file)
                    
                    df = None
                    experiment = None
                    if "run_a_content" in file_path:
                        df = exp1a_df
                        experiment = "Exp1a"
                    elif "run_b_error" in file_path:
                        df = exp1b_df
                        experiment = "Exp1b"
                        if input_source == 'script':
                            input_source = 'script_manipulated'
                    
                    cosine_sim = None
                    adherence_score = None
                    
                    if df is not None and input_source and layer is not None and llm and prompt_type:
                        matching_rows = df[
                            (df['input_source'] == input_source) & 
                            (df['layer'] == layer) & 
                            (df['llm'] == llm) & 
                            (df['prompt_type'] == prompt_type)
                        ]
                        
                        if not matching_rows.empty:
                            cosine_sim = matching_rows['cosine_similarity'].iloc[0]
                            adherence_score = matching_rows['adherence_score'].iloc[0]
                        else:
                            fallback_rows = df[
                                (df['input_source'] == input_source) & 
                                (df['layer'] == layer)
                            ]
                            if not fallback_rows.empty:
                                cosine_sim = fallback_rows['cosine_similarity'].mean()
                                adherence_score = fallback_rows['adherence_score'].mean()
                    
                    free_text_data.append({
                        'path': file_path,
                        'experiment': experiment,
                        'input_source': input_source,
                        'layer': layer,
                        'llm': llm,
                        'prompt_type': prompt_type,
                        'cosine_similarity': cosine_sim,
                        'adherence_score': adherence_score
                    })
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                pass
    
    return sorted(free_text_data, key=lambda x: x['path'])

def count_total_questions():
    base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/10_exp1")
    all_files = glob.glob(os.path.join(base_path, "**/*.txt"), recursive=True)
    
    exp1a_total = 0
    exp1b_total = 0
    
    for file_path in all_files:
        if "_question.txt" in file_path:
            if "run_a_content" in file_path:
                exp1a_total += 1
            elif "run_b_error" in file_path:
                exp1b_total += 1
    
    return exp1a_total, exp1b_total

free_text_data = find_free_text_questions_with_scores()
exp1a_total, exp1b_total = count_total_questions()

print("Free Text Questions with Scores:")
print("=" * 80)

exp1a_count = 0
exp1b_count = 0

for item in free_text_data:
    print(f"\nFile: {item['path']}")
    print(f"Experiment: {item['experiment']}")
    print(f"Input Source: {item['input_source']} | Layer: {item['layer']} | LLM: {item['llm']} | Prompt: {item['prompt_type']}")
    
    if item['cosine_similarity'] is not None:
        print(f"Cosine Similarity: {item['cosine_similarity']:.4f}")
    else:
        print("Cosine Similarity: Not found")
    
    if item['adherence_score'] is not None:
        print(f"Adherence Score: {item['adherence_score']:.4f}")
    else:
        print("Adherence Score: Not found")
    
    if item['experiment'] == "Exp1a":
        exp1a_count += 1
    elif item['experiment'] == "Exp1b":
        exp1b_count += 1

print(f"\n" + "=" * 80)
print(f"SUMMARY:")
print(f"Free Text Questions:")
print(f"  Exp1a (run_a_content): {exp1a_count}")
print(f"  Exp1b (run_b_error): {exp1b_count}")
print(f"  Total Free Text: {len(free_text_data)}")

print(f"\nTotal Questions (All Types):")
print(f"  Exp1a (run_a_content): {exp1a_total}")
print(f"  Exp1b (run_b_error): {exp1b_total}")
print(f"  Total All Questions: {exp1a_total + exp1b_total}")

print(f"\nPercentages:")
if exp1a_total > 0:
    print(f"  Exp1a Free Text: {exp1a_count}/{exp1a_total} ({100*exp1a_count/exp1a_total:.1f}%)")
if exp1b_total > 0:
    print(f"  Exp1b Free Text: {exp1b_count}/{exp1b_total} ({100*exp1b_count/exp1b_total:.1f}%)")
if (exp1a_total + exp1b_total) > 0:
    print(f"  Overall Free Text: {len(free_text_data)}/{exp1a_total + exp1b_total} ({100*len(free_text_data)/(exp1a_total + exp1b_total):.1f}%)")

print(f"\n" + "=" * 80)
print("MEAN SCORES FOR FREE TEXT QUESTIONS BY SUBEXPERIMENT:")
print("=" * 80)

free_text_df = pd.DataFrame(free_text_data)
valid_data = free_text_df.dropna(subset=['cosine_similarity', 'adherence_score'])

if not valid_data.empty:
    summary_stats = valid_data.groupby('experiment')[['cosine_similarity', 'adherence_score']].agg(['mean', 'std', 'count']).round(4)
    
    print("\nDetailed Statistics:")
    display(summary_stats)
    
    simple_summary = valid_data.groupby('experiment')[['cosine_similarity', 'adherence_score']].mean().round(4)
    simple_summary['n_questions'] = valid_data.groupby('experiment').size()
    
    print("\nSimple Summary (Mean Values):")
    display(simple_summary)
    
    overall_stats = {
        'Mean Cosine Similarity': valid_data['cosine_similarity'].mean(),
        'Mean Adherence Score': valid_data['adherence_score'].mean(),
        'Total Free Text Questions': len(valid_data)
    }
    
    print("\nOverall Free Text Questions Statistics:")
    for key, value in overall_stats.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
            
    tables["free_text_summary_by_experiment"] = simple_summary
    tables["free_text_detailed_stats"] = summary_stats
    
else:
    print("No valid free text questions with scores found!")

### 1.5: Cosine Similarity: LLM vs Input Source

In [ ]:
llm_vs_input_source_cos_sim_1a = pd.pivot_table(
    exp1a_df,
    values="cosine_similarity",
    index="llm",
    columns="input_source",
    aggfunc=["mean", "std"],
)
tables["exp1a_cosine_sim_llm_vs_input_source"] = llm_vs_input_source_cos_sim_1a
display(llm_vs_input_source_cos_sim_1a)

In [ ]:
# TODO important die tendenz zum script
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
plots['exp1a_combined_llm_vs_input_source'] = fig

llm_vs_input_source_cos_sim_1a_display = llm_vs_input_source_cos_sim_1a["mean"].copy()
llm_vs_input_source_cos_sim_1a_std = llm_vs_input_source_cos_sim_1a["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_cosine = llm_vs_input_source_cos_sim_1a_display.copy()
for i in range(len(llm_vs_input_source_cos_sim_1a_display.index)):
    for j in range(len(llm_vs_input_source_cos_sim_1a_display.columns)):
        mean_val = llm_vs_input_source_cos_sim_1a_display.iloc[i, j]
        std_val = llm_vs_input_source_cos_sim_1a_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_cosine.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_cosine.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_input_source_cos_sim_1a_display.index = llm_vs_input_source_cos_sim_1a_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_input_source_cos_sim_1a_display.columns = llm_vs_input_source_cos_sim_1a_display.columns.map({
    'script': 'Script', 'transcript': 'Transcript', 'tanenbaum': 'Tanenbaum'
})
annot_matrix_cosine.index = annot_matrix_cosine.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_cosine.columns = annot_matrix_cosine.columns.map({
    'script': 'Script', 'transcript': 'Transcript', 'tanenbaum': 'Tanenbaum'
})

sns.heatmap(llm_vs_input_source_cos_sim_1a_display, 
            annot=annot_matrix_cosine, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_input_source_cos_sim_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Cosine Similarity'},
            annot_kws={'size': 10, 'weight': 'bold'},
            ax=ax1)
ax1.set_title("Mean Cosine Similarity (-1 to 1): LLM vs Input Source (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax1.set_xlabel("Input Source", fontsize=11, labelpad=10)
ax1.set_ylabel("LLM", fontsize=11, labelpad=15)

llm_vs_input_source_adherence_1a = pd.pivot_table(
    exp1a_df,
    values="adherence_score",
    index="llm",
    columns="input_source",
    aggfunc=["mean", "std"],
)

llm_vs_input_source_adherence_1a_display = llm_vs_input_source_adherence_1a["mean"].copy()
llm_vs_input_source_adherence_1a_std = llm_vs_input_source_adherence_1a["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_adherence = llm_vs_input_source_adherence_1a_display.copy()
for i in range(len(llm_vs_input_source_adherence_1a_display.index)):
    for j in range(len(llm_vs_input_source_adherence_1a_display.columns)):
        mean_val = llm_vs_input_source_adherence_1a_display.iloc[i, j]
        std_val = llm_vs_input_source_adherence_1a_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_adherence.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_adherence.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_input_source_adherence_1a_display.index = llm_vs_input_source_adherence_1a_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_input_source_adherence_1a_display.columns = llm_vs_input_source_adherence_1a_display.columns.map({
    'script': 'Script', 'transcript': 'Transcript', 'tanenbaum': 'Tanenbaum'
})
annot_matrix_adherence.index = annot_matrix_adherence.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_adherence.columns = annot_matrix_adherence.columns.map({
    'script': 'Script', 'transcript': 'Transcript', 'tanenbaum': 'Tanenbaum'
})

sns.heatmap(llm_vs_input_source_adherence_1a_display, 
            annot=annot_matrix_adherence, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_input_source_adherence_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Adherence Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            ax=ax2)
ax2.set_title("Mean Adherence Score (0 to 1): LLM vs Input Source (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax2.set_xlabel("Input Source", fontsize=11, labelpad=10)
ax2.set_ylabel("LLM", fontsize=11, labelpad=15)

plt.tight_layout()
plt.show()


### 1.6: Cosine Similarity: LLM vs Prompt Type

In [ ]:
llm_vs_prompt_type_cos_sim_1a = pd.pivot_table(
    exp1a_df,
    values="cosine_similarity",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)
tables["exp1a_cosine_sim_llm_vs_prompt_type"] = llm_vs_prompt_type_cos_sim_1a
display(llm_vs_prompt_type_cos_sim_1a)

In [ ]:
# Combined heatmaps: LLM vs Prompt Type (Exp 1a)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
plots['exp1a_combined_llm_vs_prompt_type'] = fig

llm_vs_prompt_type_cos_sim_1a_display = llm_vs_prompt_type_cos_sim_1a["mean"].copy()
llm_vs_prompt_type_cos_sim_1a_std = llm_vs_prompt_type_cos_sim_1a["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_cosine_prompt = llm_vs_prompt_type_cos_sim_1a_display.copy()
for i in range(len(llm_vs_prompt_type_cos_sim_1a_display.index)):
    for j in range(len(llm_vs_prompt_type_cos_sim_1a_display.columns)):
        mean_val = llm_vs_prompt_type_cos_sim_1a_display.iloc[i, j]
        std_val = llm_vs_prompt_type_cos_sim_1a_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_cosine_prompt.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_cosine_prompt.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_prompt_type_cos_sim_1a_display.index = llm_vs_prompt_type_cos_sim_1a_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_prompt_type_cos_sim_1a_display.columns = llm_vs_prompt_type_cos_sim_1a_display.columns.map({
    'common': 'Common', 'complex': 'Complex'
})
annot_matrix_cosine_prompt.index = annot_matrix_cosine_prompt.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_cosine_prompt.columns = annot_matrix_cosine_prompt.columns.map({
    'common': 'Common', 'complex': 'Complex'
})

sns.heatmap(llm_vs_prompt_type_cos_sim_1a_display, 
            annot=annot_matrix_cosine_prompt, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_prompt_type_cos_sim_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Cosine Similarity'},
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax1)
ax1.set_title("Mean Cosine Similarity (-1 to 1): LLM vs Prompt Type (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax1.set_xlabel("Prompt Type", fontsize=11, labelpad=10)
ax1.set_ylabel("LLM", fontsize=11, labelpad=15)

llm_vs_prompt_type_adherence_1a = pd.pivot_table(
    exp1a_df,
    values="adherence_score",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)

llm_vs_prompt_type_adherence_1a_display = llm_vs_prompt_type_adherence_1a["mean"].copy()
llm_vs_prompt_type_adherence_1a_std = llm_vs_prompt_type_adherence_1a["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_adherence_prompt = llm_vs_prompt_type_adherence_1a_display.copy()
for i in range(len(llm_vs_prompt_type_adherence_1a_display.index)):
    for j in range(len(llm_vs_prompt_type_adherence_1a_display.columns)):
        mean_val = llm_vs_prompt_type_adherence_1a_display.iloc[i, j]
        std_val = llm_vs_prompt_type_adherence_1a_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_adherence_prompt.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_adherence_prompt.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_prompt_type_adherence_1a_display.index = llm_vs_prompt_type_adherence_1a_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_prompt_type_adherence_1a_display.columns = llm_vs_prompt_type_adherence_1a_display.columns.map({
    'common': 'Common', 'complex': 'Complex'
})
annot_matrix_adherence_prompt.index = annot_matrix_adherence_prompt.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_adherence_prompt.columns = annot_matrix_adherence_prompt.columns.map({
    'common': 'Common', 'complex': 'Complex'
})

sns.heatmap(llm_vs_prompt_type_adherence_1a_display, 
            annot=annot_matrix_adherence_prompt, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_prompt_type_adherence_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Adherence Score'},
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax2)
ax2.set_title("Mean Adherence Score (0 to 1): LLM vs Prompt Type (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax2.set_xlabel("Prompt Type", fontsize=11, labelpad=10)
ax2.set_ylabel("LLM", fontsize=11, labelpad=15)

plt.tight_layout()
plt.show()


### 1.7: Adherence Score: LLM vs Prompt Type

In [ ]:
llm_vs_prompt_type_adherence_1a = pd.pivot_table(
    exp1a_df,
    values="adherence_score",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)
tables["exp1a_adherence_llm_vs_prompt_type"] = llm_vs_prompt_type_adherence_1a
display(llm_vs_prompt_type_adherence_1a)

### 1.8: Adherence Score: LLM vs Input Source

In [ ]:
llm_vs_input_source_adherence_1a = pd.pivot_table(
    exp1a_df,
    values="adherence_score",
    index="llm",
    columns="input_source",
    aggfunc=["mean", "std"],
)
tables["exp1a_adherence_llm_vs_input_source"] = llm_vs_input_source_adherence_1a
display(llm_vs_input_source_adherence_1a)

### 1.9: Distribution Plots

In [ ]:
# Boxplot for cosine_similarity and adherence_score by LLM
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1a_dist_by_llm'] = fig

create_seaborn_boxplot(exp1a_df, 'llm', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by LLM (Exp 1a)',
                      'Cosine Similarity', 'LLM')

create_seaborn_boxplot(exp1a_df, 'llm', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by LLM (Exp 1a)', 
                      'Adherence Score', 'LLM')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplot for cosine_similarity and adherence_score by LLM
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1b_dist_by_llm'] = fig

create_seaborn_boxplot(exp1b_df, 'llm', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by LLM (Exp 1b)',
                      'Cosine Similarity', 'LLM')

create_seaborn_boxplot(exp1b_df, 'llm', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by LLM (Exp 1b)', 
                      'Adherence Score', 'LLM')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplot for cosine_similarity and adherence_score by input_source
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1a_dist_by_input_source'] = fig

create_seaborn_boxplot(exp1a_df, 'input_source', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Input Source (Exp 1a)',
                      'Cosine Similarity', 'Input Source')

create_seaborn_boxplot(exp1a_df, 'input_source', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Input Source (Exp 1a)', 
                      'Adherence Score', 'Input Source')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for cosine_similarity and adherence_score by input_source
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1b_dist_by_input_source'] = fig

create_seaborn_boxplot(exp1b_df, 'input_source', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Input Source (Exp 1b)',
                      'Cosine Similarity', 'Input Source')

create_seaborn_boxplot(exp1b_df, 'input_source', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Input Source (Exp 1b)', 
                      'Adherence Score', 'Input Source')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for cosine_similarity and adherence_score by prompt_type
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1a_dist_by_prompt_type'] = fig

create_seaborn_boxplot(exp1a_df, 'prompt_type', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Prompt Type (Exp 1a)',
                      'Cosine Similarity', 'Prompt Type')

create_seaborn_boxplot(exp1a_df, 'prompt_type', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Prompt Type (Exp 1a)', 
                      'Adherence Score', 'Prompt Type')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for cosine_similarity and adherence_score by prompt_type
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1b_dist_by_prompt_type'] = fig

create_seaborn_boxplot(exp1b_df, 'prompt_type', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Prompt Type (Exp 1b)',
                      'Cosine Similarity', 'Prompt Type')

create_seaborn_boxplot(exp1b_df, 'prompt_type', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Prompt Type (Exp 1b)', 
                      'Adherence Score', 'Prompt Type')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for cosine_similarity and adherence_score by layer
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1a_dist_by_layer'] = fig

create_seaborn_boxplot(exp1a_df, 'layer', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Layer (Exp 1a)',
                      'Cosine Similarity', 'Layer')

create_seaborn_boxplot(exp1a_df, 'layer', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Layer (Exp 1a)', 
                      'Adherence Score', 'Layer')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for cosine_similarity and adherence_score by layer
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1b_dist_by_layer'] = fig

create_seaborn_boxplot(exp1b_df, 'layer', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Distribution by Layer (Exp 1b)',
                      'Cosine Similarity', 'Layer')

create_seaborn_boxplot(exp1b_df, 'layer', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Distribution by Layer (Exp 1b)', 
                      'Adherence Score', 'Layer')

plt.tight_layout()
plt.show()

In [ ]:
# # Boxplot for cosine_similarity and adherence_score by question_type
# fig, ax = plt.subplots(1, 2, figsize=(15, 7))
# plots['exp1a_dist_by_question_type'] = fig

# sns.boxplot(x='question_type', y='cosine_similarity', data=exp1a_df, ax=ax[0], color='lightcoral')
# ax[0].set_title('Cosine Similarity Distribution by Question Type (Exp 1a)')
# ax[0].tick_params(axis='x', rotation=45)

# sns.boxplot(x='question_type', y='adherence_score', data=exp1a_df, ax=ax[1], color='lightcoral')
# ax[1].set_title('Adherence Score Distribution by Question Type (Exp 1a)')
# ax[1].tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.show()

In [ ]:
# # Boxplot for cosine_similarity and adherence_score by question_type
# fig, ax = plt.subplots(1, 2, figsize=(15, 7))
# plots['exp1b_dist_by_question_type'] = fig

# sns.boxplot(x='question_type', y='cosine_similarity', data=exp1b_df, ax=ax[0], color='lightcoral')
# ax[0].set_title('Cosine Similarity Distribution by Question Type (Exp 1b)')
# ax[0].tick_params(axis='x', rotation=45)

# sns.boxplot(x='question_type', y='adherence_score', data=exp1b_df, ax=ax[1], color='lightcoral')
# ax[1].set_title('Adherence Score Distribution by Question Type (Exp 1b)')
# ax[1].tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.show()

In [ ]:
exp1a_temp = exp1a_df.copy()
exp1b_temp = exp1b_df.copy()

exp1a_temp['experiment'] = '1a'
exp1b_temp['experiment'] = '1b'

combined_df = pd.concat([exp1a_temp, exp1b_temp])

fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plots['exp1_comparison'] = fig

create_seaborn_boxplot(combined_df, 'experiment', 'cosine_similarity', ax[0], 
                      'Cosine Similarity (-1 to 1) Comparison (Exp 1a vs 1b)',
                      'Cosine Similarity', 'Experiment')

create_seaborn_boxplot(combined_df, 'experiment', 'adherence_score', ax[1],
                      'Adherence Score (0 to 1) Comparison (Exp 1a vs 1b)', 
                      'Adherence Score', 'Experiment')

plt.tight_layout()
plt.show()

# Updated combined heatmap section
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# For cosine similarity
llm_vs_input_source_cos_sim_1a_std_final = llm_vs_input_source_cos_sim_1a["std"].copy()
annot_matrix_cosine_final = llm_vs_input_source_cos_sim_1a_display.copy()
for i in range(len(llm_vs_input_source_cos_sim_1a_display.index)):
    for j in range(len(llm_vs_input_source_cos_sim_1a_display.columns)):
        mean_val = llm_vs_input_source_cos_sim_1a_display.iloc[i, j]
        std_val = llm_vs_input_source_cos_sim_1a_std_final.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_cosine_final.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_cosine_final.iloc[i, j] = f"{mean_val:.2f}"

sns.heatmap(llm_vs_input_source_cos_sim_1a_display, 
            annot=annot_matrix_cosine_final, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_input_source_cos_sim_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Cosine Similarity'},
            annot_kws={'size': 10, 'weight': 'bold'},
            ax=ax1)
ax1.set_title("Mean Cosine Similarity (-1 to 1): LLM vs Input Source (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax1.set_ylabel("LLM", fontsize=11, labelpad=15)
ax1.set_xlabel("Input Source", fontsize=11, labelpad=10)

# For adherence score
llm_vs_input_source_adherence_1a_std_final = llm_vs_input_source_adherence_1a["std"].copy()
annot_matrix_adherence_final = llm_vs_input_source_adherence_1a_display.copy()
for i in range(len(llm_vs_input_source_adherence_1a_display.index)):
    for j in range(len(llm_vs_input_source_adherence_1a_display.columns)):
        mean_val = llm_vs_input_source_adherence_1a_display.iloc[i, j]
        std_val = llm_vs_input_source_adherence_1a_std_final.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_adherence_final.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_adherence_final.iloc[i, j] = f"{mean_val:.2f}"

sns.heatmap(llm_vs_input_source_adherence_1a_display, 
            annot=annot_matrix_adherence_final, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_input_source_adherence_1a_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Adherence Score'},
            annot_kws={'size': 10, 'weight': 'bold'},
            ax=ax2)
ax2.set_title("Mean Adherence Score (0 to 1): LLM vs Input Source (Exp 1a)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax2.set_ylabel("LLM", fontsize=11, labelpad=15)
ax2.set_xlabel("Input Source", fontsize=11, labelpad=10)

plt.tight_layout()
plt.show()


## Experiment 1b

### 2.1: Metrics by LLM

In [ ]:
llm_metrics_1b = exp1b_df.groupby("llm")[["cosine_similarity", "adherence_score"]].agg(
    ["mean", "std"]
)
tables["exp1b_metrics_by_llm"] = llm_metrics_1b
display(llm_metrics_1b)


### 2.2: Metrics by Input Source

In [ ]:
input_source_metrics_1b = exp1b_df.groupby("input_source")[
    ["cosine_similarity", "adherence_score"]
].agg(["mean", "std"])
tables["exp1b_metrics_by_input_source"] = input_source_metrics_1b
display(input_source_metrics_1b)

### 2.3: Metrics by Prompt Type

In [ ]:
prompt_type_metrics_1b = exp1b_df.groupby("prompt_type")[
    ["cosine_similarity", "adherence_score"]
].agg(["mean", "std"])
tables["exp1b_metrics_by_prompt_type"] = prompt_type_metrics_1b
display(prompt_type_metrics_1b)

### 2.4: Metrics by Layer

In [ ]:
layer_metrics_1b = exp1b_df.groupby("layer")[["cosine_similarity", "adherence_score"]].agg(
    ["mean", "std"]
)
tables["exp1b_metrics_by_layer"] = layer_metrics_1b
display(layer_metrics_1b)

### 2.5: Cosine Similarity: LLM vs Input Source

In [ ]:
# TODO important
llm_vs_input_source_cos_sim_1b = pd.pivot_table(
    exp1b_df,
    values="cosine_similarity",
    index="llm",
    columns="input_source",
    aggfunc=["mean", "std"],
)
tables["exp1b_cosine_sim_llm_vs_input_source"] = llm_vs_input_source_cos_sim_1b
display(llm_vs_input_source_cos_sim_1b)

In [ ]:

# fig, ax = plt.subplots(figsize=(10, 6))
# plots['exp1b_cosine_sim_llm_vs_input_source'] = fig
# sns.heatmap(llm_vs_input_source_cos_sim_1b["mean"], annot=True, fmt=".2f", cmap="viridis", ax=ax)
# ax.set_title("Mean Cosine Similarity: LLM vs Input Source (Exp 1b)")
# plt.show()


### 2.6: Cosine Similarity: LLM vs Prompt Type

In [ ]:
# TODO important
llm_vs_prompt_type_cos_sim_1b = pd.pivot_table(
    exp1b_df,
    values="cosine_similarity",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)
tables["exp1b_cosine_sim_llm_vs_prompt_type"] = llm_vs_prompt_type_cos_sim_1b
display(llm_vs_prompt_type_cos_sim_1b)

In [ ]:
# Combined heatmaps: LLM vs Prompt Type (Exp 1b)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
plots['exp1b_combined_llm_vs_prompt_type'] = fig

llm_vs_prompt_type_cos_sim_1b_display = llm_vs_prompt_type_cos_sim_1b["mean"].copy()
llm_vs_prompt_type_cos_sim_1b_std = llm_vs_prompt_type_cos_sim_1b["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_cosine_prompt_1b = llm_vs_prompt_type_cos_sim_1b_display.copy()
for i in range(len(llm_vs_prompt_type_cos_sim_1b_display.index)):
    for j in range(len(llm_vs_prompt_type_cos_sim_1b_display.columns)):
        mean_val = llm_vs_prompt_type_cos_sim_1b_display.iloc[i, j]
        std_val = llm_vs_prompt_type_cos_sim_1b_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_cosine_prompt_1b.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_cosine_prompt_1b.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_prompt_type_cos_sim_1b_display.index = llm_vs_prompt_type_cos_sim_1b_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_prompt_type_cos_sim_1b_display.columns = llm_vs_prompt_type_cos_sim_1b_display.columns.map({
    'common': 'Common', 'complex': 'Complex'
})
annot_matrix_cosine_prompt_1b.index = annot_matrix_cosine_prompt_1b.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_cosine_prompt_1b.columns = annot_matrix_cosine_prompt_1b.columns.map({
    'common': 'Common', 'complex': 'Complex'
})

sns.heatmap(llm_vs_prompt_type_cos_sim_1b_display, 
            annot=annot_matrix_cosine_prompt_1b, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_prompt_type_cos_sim_1b_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Cosine Similarity'},
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax1)
ax1.set_title("Mean Cosine Similarity (-1 to 1): LLM vs Prompt Type (Exp 1b)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax1.set_xlabel("Prompt Type", fontsize=11, labelpad=10)
ax1.set_ylabel("LLM", fontsize=11, labelpad=15)

llm_vs_prompt_type_adherence_1b = pd.pivot_table(
    exp1b_df,
    values="adherence_score",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)

llm_vs_prompt_type_adherence_1b_display = llm_vs_prompt_type_adherence_1b["mean"].copy()
llm_vs_prompt_type_adherence_1b_std = llm_vs_prompt_type_adherence_1b["std"].copy()

# Create annotation matrix with mean and std
annot_matrix_adherence_prompt_1b = llm_vs_prompt_type_adherence_1b_display.copy()
for i in range(len(llm_vs_prompt_type_adherence_1b_display.index)):
    for j in range(len(llm_vs_prompt_type_adherence_1b_display.columns)):
        mean_val = llm_vs_prompt_type_adherence_1b_display.iloc[i, j]
        std_val = llm_vs_prompt_type_adherence_1b_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_adherence_prompt_1b.iloc[i, j] = f"{mean_val:.2f}\n({std_val:.2f})"
        elif pd.notna(mean_val):
            annot_matrix_adherence_prompt_1b.iloc[i, j] = f"{mean_val:.2f}"

llm_vs_prompt_type_adherence_1b_display.index = llm_vs_prompt_type_adherence_1b_display.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
llm_vs_prompt_type_adherence_1b_display.columns = llm_vs_prompt_type_adherence_1b_display.columns.map({
    'common': 'Common', 'complex': 'Complex'
})
annot_matrix_adherence_prompt_1b.index = annot_matrix_adherence_prompt_1b.index.map({
    'anthropic': 'Anthropic', 'openai': 'OpenAI', 'google': 'Google', 'deepseek': 'DeepSeek'
})
annot_matrix_adherence_prompt_1b.columns = annot_matrix_adherence_prompt_1b.columns.map({
    'common': 'Common', 'complex': 'Complex'
})

sns.heatmap(llm_vs_prompt_type_adherence_1b_display, 
            annot=annot_matrix_adherence_prompt_1b, 
            fmt='', 
            cmap='RdYlBu_r',
            center=llm_vs_prompt_type_adherence_1b_display.mean().mean(),
            square=True,
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Adherence Score'},
            annot_kws={'size': 11, 'weight': 'bold'},
            ax=ax2)
ax2.set_title("Mean Adherence Score (0 to 1): LLM vs Prompt Type (Exp 1b)\nValues: Mean (Std)", 
              fontsize=12, fontweight='bold', pad=15)
ax2.set_xlabel("Prompt Type", fontsize=11, labelpad=10)
ax2.set_ylabel("LLM", fontsize=11, labelpad=15)

plt.tight_layout()
plt.show()


### 2.7: Adherence Score: LLM vs Prompt Type

In [ ]:
# TODO important
llm_vs_prompt_type_adherence_1b = pd.pivot_table(
    exp1b_df,
    values="adherence_score",
    index="llm",
    columns="prompt_type",
    aggfunc=["mean", "std"],
)
tables["exp1b_adherence_llm_vs_prompt_type"] = llm_vs_prompt_type_adherence_1b
display(llm_vs_prompt_type_adherence_1b)

In [ ]:
# TODO can be important
# Diese Heatmap ist jetzt im kombinierten Plot oben enthalten
# fig, ax = plt.subplots(figsize=(10, 6))
# plots['exp1b_adherence_llm_vs_prompt_type'] = fig
# sns.heatmap(llm_vs_prompt_type_adherence_1b["mean"], annot=True, fmt=".2f", cmap="viridis", ax=ax)
# ax.set_title("Mean Adherence Score: LLM vs Prompt Type (Exp 1b)")
# plt.show()


### 2.8: Adherence Score: LLM vs Input Source

In [ ]:
#TODO important
llm_vs_input_source_adherence_1b = pd.pivot_table(
    exp1b_df,
    values="adherence_score",
    index="llm",
    columns="input_source",
    aggfunc=["mean", "std"],
)
tables["exp1b_adherence_llm_vs_input_source"] = llm_vs_input_source_adherence_1b
display(llm_vs_input_source_adherence_1b)

### Including a Run without a Source

In [ ]:
## Experiment 1a No Source

if exp1a_no_source_df is not None:
    print("Creating plots for exp1a_no_source data...")
    
    # Plot 1: Distribution by LLM for no_source
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    plots['exp1a_no_source_dist_by_llm'] = fig

    create_seaborn_boxplot(exp1a_no_source_df, 'llm', 'cosine_similarity', ax[0], 
                          'Cosine Similarity (-1 to 1) Distribution by LLM (Exp 1a No Source)',
                          'Cosine Similarity', 'LLM')

    create_seaborn_boxplot(exp1a_no_source_df, 'llm', 'adherence_score', ax[1],
                          'Adherence Score (0 to 1) Distribution by LLM (Exp 1a No Source)', 
                          'Adherence Score', 'LLM')

    plt.tight_layout()
    plt.show()
    
    # Plot 2: Comparison between all input sources (including no_source)
    exp1a_fresh = pd.read_csv(exp1a_path)
    exp1a_fresh["adherence_score"] = exp1a_fresh[
        ["adherence_score_openai", "adherence_score_anthropic"]
    ].mean(axis=1)
    exp1a_fresh = exp1a_fresh.drop(
        columns=["adherence_score_openai", "adherence_score_anthropic"]
    )
    
    print("Creating plots for exp1a_no_source data...")

    # Combine the data for plotting
    exp1a_combined_df = pd.concat([exp1a_fresh, exp1a_no_source_df], ignore_index=True)

    # Calculate means for each input source from the separate datasets
    exp1a_means = exp1a_fresh.groupby('input_source')[['cosine_similarity', 'adherence_score']].mean()
    no_source_means = exp1a_no_source_df.groupby('input_source')[['cosine_similarity', 'adherence_score']].mean()

    # Combine all means
    all_means = pd.concat([exp1a_means, no_source_means])

    print("Means for cosine similarity:")
    print(all_means['cosine_similarity'])
    print("\nMeans for adherence score:")
    print(all_means['adherence_score'])

    # Create figure for input source comparison
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    plots['exp1a_comparison_with_no_source'] = fig

    # Plot cosine similarity by input source with custom means
    create_seaborn_boxplot(exp1a_combined_df, 'input_source', 'cosine_similarity', ax[0], 
                          'Cosine Similarity (-1 to 1) by Input Source (Exp 1a with No Source)', 
                          'Cosine Similarity', 'Input Source',
                          custom_means=all_means['cosine_similarity'])

    # Plot adherence score by input source with custom means
    create_seaborn_boxplot(exp1a_combined_df, 'input_source', 'adherence_score', ax[1],
                          'Adherence Score (0 to 1) by Input Source (Exp 1a with No Source)', 
                          'Adherence Score', 'Input Source',
                          custom_means=all_means['adherence_score'])

    plt.tight_layout()
    plt.show()
    
    print("exp1a_no_source plots created successfully!")
else:
    print("Skipping exp1a_no_source plots - data not available.")

## Save all tables and plots

In [ ]:
def save_all_tables_and_plots():
    for name, df in tables.items():
        if name.startswith("exp1a_no_source"):
            output_dir = output_dir_exp1a_no_source
            table_name = name.replace("exp1a_no_source_", "")
        elif name.startswith("exp1a"):
            output_dir = output_dir_exp1a
            table_name = name.replace("exp1a_", "")
        elif name.startswith("exp1b"):
            output_dir = output_dir_exp1b
            table_name = name.replace("exp1b_", "")
        else:
            if "exp1" in name:
                base_filename = os.path.join(output_dir_exp1a, name)
                csv_path = f"{base_filename}.csv"
                df.to_csv(csv_path)
                print(f"Saved {csv_path}")
            continue

        base_filename = os.path.join(output_dir, table_name)
        
        csv_path = f"{base_filename}.csv"
        df.to_csv(csv_path)
        print(f"Saved {csv_path}")

    for name, fig in plots.items():
        if name.startswith("exp1a_no_source") or "no_source" in name:
            output_dir = output_plot_dir_exp1a_no_source
            plot_name = name.replace("exp1a_no_source_", "").replace("exp1a_", "")
        elif name.startswith("exp1a"):
            output_dir = output_plot_dir_exp1a
            plot_name = name.replace("exp1a_", "")
        elif name.startswith("exp1b"):
            output_dir = output_plot_dir_exp1b
            plot_name = name.replace("exp1b_", "")
        else:
            if "exp1" in name:
                base_filename = os.path.join(output_plot_dir_exp1a, name)
                png_path = f"{base_filename}.png"
                fig.savefig(png_path, bbox_inches='tight')
                print(f"Saved {png_path}")
            continue
        
        base_filename = os.path.join(output_dir, plot_name)
        
        png_path = f"{base_filename}.png"
        fig.savefig(png_path, bbox_inches='tight')
        print(f"Saved {png_path}")

save_all_tables_and_plots()